# Day 41 / 42: Deploying an ML Model with FastAPI

**42 Days of ML Challenge**

A model that scores 95% accuracy in a notebook is worth nothing until it can answer a real request. Today you'll take a trained model and wrap it in a REST API using FastAPI, the same pattern used in real production ML services.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week6_production/day41_fastapi_deployment/day41_notebook.ipynb)

---

### What You'll Learn
- Why a trained model isn't useful until it's served
- Model serialization with `joblib`
- Wrapping inference in a FastAPI endpoint
- Input validation with Pydantic (and why it matters in production)
- Testing an API without needing a separate running server
- What changes when this goes from your laptop to real production

---


## Setup

Install the packages needed for this notebook. If you're running this in Google Colab, run this cell first. If you're in Jupyter Lab/Notebook locally, make sure these are installed in your environment.

In [1]:
!pip install fastapi uvicorn scikit-learn pandas joblib httpx -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

---
## Part 1: Train and Save a Model

We're reusing the Titanic classifier from Week 3 (Day 15). The point of today isn't the model, it's what happens *after* the model is trained. If you already have a saved model from Day 15, feel free to load it directly instead of retraining here.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
df = df[FEATURES + ["Survived"]].copy()

# Same cleaning approach as Week 1: median imputation for Age, encode Sex
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

X = df[FEATURES]
y = df["Survived"]

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (891, 7)


,Pclass,Sex,Age,SibSp,Parch,Fare,Survived
0,3,0,22.0,1,0,7.2500,0
1,1,1,38.0,1,0,71.2833,1
2,3,1,26.0,0,0,7.9250,1
3,1,1,35.0,1,0,53.1000,1
4,3,0,35.0,0,0,8.0500,0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print()
print(classification_report(y_test, preds))

Accuracy: 0.8045

              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



### Serialize the model

This is the step every ML course skips. A model sitting in your notebook's memory is useless the moment you close the notebook. `joblib` saves the trained model (and the scaler, which is just as important) to disk so it can be loaded by a completely separate process, like an API server.

**Production note:** always save your preprocessing objects (scaler, encoder) alongside the model. If you only save the model and forget the scaler, your API will silently make wrong predictions on unscaled input. This is a real, documented class of production bugs.

In [4]:
joblib.dump(model, "titanic_model.joblib")
joblib.dump(scaler, "titanic_scaler.joblib")

print("Saved: titanic_model.joblib")
print("Saved: titanic_scaler.joblib")

Saved: titanic_model.joblib
Saved: titanic_scaler.joblib


---
## Part 2: Wrap the Model in a FastAPI App

Three things happen here:
1. **Load** the saved model and scaler (simulating a fresh process starting up, the way a real server would)
2. **Define the input schema** with Pydantic, this is what protects your API from bad requests
3. **Define the `/predict` endpoint** that takes passenger details and returns a prediction

This is the exact pattern used in production: the model is trained once, offline, and the API only ever loads it and runs `.predict()`. It never retrains inside a request.

In [5]:
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Load the saved artifacts (simulates what happens when the API server starts up)
model = joblib.load("titanic_model.joblib")
scaler = joblib.load("titanic_scaler.joblib")
FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]

app = FastAPI(title="Titanic Survival Predictor", version="1.0")


class PassengerInput(BaseModel):
    Pclass: int = Field(..., ge=1, le=3, description="Ticket class: 1, 2, or 3")
    Sex: int = Field(..., ge=0, le=1, description="0 = male, 1 = female")
    Age: float = Field(..., ge=0, le=100)
    SibSp: int = Field(..., ge=0, description="Siblings/spouses aboard")
    Parch: int = Field(..., ge=0, description="Parents/children aboard")
    Fare: float = Field(..., ge=0)


class PredictionOutput(BaseModel):
    survived: int
    survival_probability: float
    risk_tier: str


@app.get("/")
def root():
    return {"status": "ok", "message": "Titanic Survival Predictor API"}


@app.post("/predict", response_model=PredictionOutput)
def predict(passenger: PassengerInput):
    row = pd.DataFrame([passenger.model_dump()], columns=FEATURES)
    row_scaled = scaler.transform(row)

    pred = model.predict(row_scaled)[0]
    prob = model.predict_proba(row_scaled)[0][1]

    if prob >= 0.66:
        risk_tier = "high"
    elif prob >= 0.33:
        risk_tier = "medium"
    else:
        risk_tier = "low"

    return PredictionOutput(
        survived=int(pred),
        survival_probability=round(float(prob), 3),
        risk_tier=risk_tier,
    )

print("FastAPI app defined.")

FastAPI app defined.


### Why Pydantic matters here

Without the `PassengerInput` schema, if someone sends `"Pclass": "first class"` instead of `"Pclass": 1`, your model's `.predict()` call would either crash with a cryptic sklearn error, or worse, silently produce a garbage prediction. Pydantic validates the request *before* it ever reaches your model, and returns a clear error instead of taking down your service.

This is the difference between a notebook demo and something you can actually put in production.

---
## Part 3: Test the API (No Server Required)

Normally you'd run `uvicorn main:app --reload` in a terminal and hit the API with `requests` or Postman. That's awkward inside a notebook, especially in Colab.

FastAPI's `TestClient` lets you test the exact same app, routes, validation, everything, without spinning up a separate server process. This is also genuinely how you'd write automated tests for this API in a real project.

In [6]:
from fastapi.testclient import TestClient

client = TestClient(app)

# Test 1: health check
response = client.get("/")
print("GET / ->", response.status_code)
print(response.json())

GET / -> 200
{'status': 'ok', 'message': 'Titanic Survival Predictor API'}


/usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [7]:
# Test 2: a valid prediction request
payload = {
    "Pclass": 1,
    "Sex": 1,      # female
    "Age": 29,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 100.0
}

response = client.post("/predict", json=payload)
print("POST /predict ->", response.status_code)
print(response.json())

POST /predict -> 200
{'survived': 1, 'survival_probability': 0.948, 'risk_tier': 'high'}


In [8]:
# Test 3: another passenger, different profile
payload_2 = {
    "Pclass": 3,
    "Sex": 0,      # male
    "Age": 40,
    "SibSp": 1,
    "Parch": 0,
    "Fare": 8.5
}

response = client.post("/predict", json=payload_2)
print("POST /predict ->", response.status_code)
print(response.json())

POST /predict -> 200
{'survived': 0, 'survival_probability': 0.051, 'risk_tier': 'low'}


### Now let's break it on purpose

This is the production problem from today's post: what happens when the API receives bad input? Watch how Pydantic handles it instead of crashing the whole service.

In [9]:
# Test 4: invalid input — Pclass sent as a string instead of an int
bad_payload = {
    "Pclass": "first class",   # wrong type, on purpose
    "Sex": 1,
    "Age": 29,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 100.0
}

response = client.post("/predict", json=bad_payload)
print("POST /predict (bad input) ->", response.status_code)
print(response.json())

print()
print("Notice: the API returned a 422 with a clear error message.")
print("It did NOT crash, and it did NOT silently return a wrong prediction.")
print("This is exactly what Pydantic validation buys you in production.")

POST /predict (bad input) -> 422
{'detail': [{'type': 'int_parsing', 'loc': ['body', 'Pclass'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'first class'}]}

Notice: the API returned a 422 with a clear error message.
It did NOT crash, and it did NOT silently return a wrong prediction.
This is exactly what Pydantic validation buys you in production.


In [10]:
# Test 5: out-of-range input — Pclass = 7 doesn't exist, Age = -5 is impossible
bad_payload_2 = {
    "Pclass": 7,
    "Sex": 1,
    "Age": -5,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 100.0
}

response = client.post("/predict", json=bad_payload_2)
print("POST /predict (out of range) ->", response.status_code)
print(response.json())

POST /predict (out of range) -> 422
{'detail': [{'type': 'less_than_equal', 'loc': ['body', 'Pclass'], 'msg': 'Input should be less than or equal to 3', 'input': 7, 'ctx': {'le': 3}}, {'type': 'greater_than_equal', 'loc': ['body', 'Age'], 'msg': 'Input should be greater than or equal to 0', 'input': -5, 'ctx': {'ge': 0.0}}]}


---
## Part 4: Running This for Real (Outside the Notebook)

Everything above runs inside the notebook using `TestClient`. To actually serve this API so other applications (a website, a mobile app, another service) can call it, you'd run it as a standalone script.

**This part will NOT run inside Colab or Jupyter** — it needs to run from a terminal. It's here so you have the real deployment code.

In [11]:
# Save this as main.py and run it separately with: uvicorn main:app --reload

fastapi_app_code = """
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

model = joblib.load("titanic_model.joblib")
scaler = joblib.load("titanic_scaler.joblib")
FEATURES = [\"Pclass\", \"Sex\", \"Age\", \"SibSp\", \"Parch\", \"Fare\"]

app = FastAPI(title="Titanic Survival Predictor", version="1.0")

class PassengerInput(BaseModel):
    Pclass: int = Field(..., ge=1, le=3)
    Sex: int = Field(..., ge=0, le=1)
    Age: float = Field(..., ge=0, le=100)
    SibSp: int = Field(..., ge=0)
    Parch: int = Field(..., ge=0)
    Fare: float = Field(..., ge=0)

class PredictionOutput(BaseModel):
    survived: int
    survival_probability: float
    risk_tier: str

@app.get("/")
def root():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOutput)
def predict(passenger: PassengerInput):
    row = pd.DataFrame([passenger.model_dump()], columns=FEATURES)
    row_scaled = scaler.transform(row)
    pred = model.predict(row_scaled)[0]
    prob = model.predict_proba(row_scaled)[0][1]
    risk_tier = "high" if prob >= 0.66 else "medium" if prob >= 0.33 else "low"
    return PredictionOutput(survived=int(pred), survival_probability=round(float(prob), 3), risk_tier=risk_tier)
"""

with open("main.py", "w") as f:
    f.write(fastapi_app_code)

print("main.py written.")
print("Run from a terminal in the same folder with:")
print("    uvicorn main:app --reload")
print()
print("Then open http://127.0.0.1:8000/docs in your browser.")
print("FastAPI auto-generates a full interactive API tester there — no Postman needed.")

main.py written.
Run from a terminal in the same folder with:
    uvicorn main:app --reload

Then open http://127.0.0.1:8000/docs in your browser.
FastAPI auto-generates a full interactive API tester there — no Postman needed.


### What you'd see at `/docs`

Once running locally, FastAPI automatically generates interactive API documentation at `http://127.0.0.1:8000/docs` (Swagger UI). You can send test requests directly from the browser, see the exact schema your API expects, and share this link with anyone who needs to integrate with your model. No separate documentation to write and maintain, it's generated from your Pydantic models.

This is genuinely useful in interviews too, it's proof you can hand someone a working, documented API, not just a script.

---
## Real World Problem

A fintech MLE deploys a credit scoring model as an API. It works fine in testing. In production, a downstream service occasionally sends `income` as a string like `"50,000"` instead of a number. Without input validation, this either crashes the service or, worse, gets silently coerced into something wrong and produces a garbage prediction that nobody notices for weeks.

The fix isn't a smarter model. It's the exact pattern in this notebook: strict schema validation at the API boundary, before the request ever reaches the model. In production ML, this boundary is where most real incidents happen, not inside the model itself.

---
## Interview Corner

**Q: How would you deploy a trained ML model as a service?**

**What they're testing:** Whether you understand the deployment pattern, not just whether you can call `.predict()`.

**Answer direction:**
- Serialize the trained model (and any preprocessing objects) with joblib or pickle
- Wrap inference in a REST endpoint using a framework like FastAPI or Flask
- Validate incoming requests with a schema (Pydantic) before they reach the model
- The model is trained once, offline; the API only loads and serves it, it never retrains inside a request
- Mention statelessness: each request should be independent, so the service can be scaled horizontally
- Bonus points: mention containerizing it with Docker and that the model file path should come from an environment variable, not be hardcoded

---
## ML Spotlight

**FastAPI's automatic interactive docs.** Every FastAPI app gets a `/docs` endpoint for free, generated directly from your Pydantic models. It's not just a nice-to-have, it means anyone integrating with your API (a frontend developer, another team, an interviewer looking at your project) can see exactly what your API expects and test it live, with zero extra documentation work from you.

Docs: [fastapi.tiangolo.com](https://fastapi.tiangolo.com)

---
## Key Takeaways

- A trained model is not a product. A served model is.
- Always save preprocessing objects (scaler, encoder) alongside the model, not just the model itself.
- Input validation at the API boundary prevents entire classes of production incidents.
- `TestClient` lets you test a FastAPI app fully without running a separate server, useful for notebooks and automated tests alike.
- The model never retrains inside a request. Training and serving are separate stages.

---

### What's Next

**Day 42: MLOps Basics** — experiment tracking, model versioning, and how teams know when a deployed model's assumptions have quietly stopped holding.

**Day 43: The Capstone** — full pipeline from raw data to a monitored production API, tying every phase of this series together into one resume-ready project.

Full code: [github.com/VaishnaviJagtap18/42-days-aiml-challenge](https://github.com/VaishnaviJagtap18/42-days-aiml-challenge)

#42DaysOfML
